In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

#nama kolom
col_names = [
    'engine_id', 'cycle',
    'setting_1', 'setting_2', 'setting_3',
    's1', 's2', 's3', 's4', 's5',
    's6', 's7', 's8', 's9', 's10',
    's11', 's12', 's13', 's14', 's15',
    's16', 's17', 's18', 's19', 's20', 's21'
]

train = pd.read_csv("../data/train_FD001.txt", sep=r'\s+', header=None, names=col_names)
test = pd.read_csv("../data/test_FD001.txt", sep=r'\s+', header=None, names=col_names)
rul = pd.read_csv("../data/RUL_FD001.txt", sep=r'\s+', header=None, names=['RUL'])

#tambah kolom RUL dan label ke train
rul_train = train.groupby('engine_id')['cycle'].max().reset_index()
rul_train.columns = ['engine_id', 'max_cycle']
train = train.merge(rul_train, on='engine_id')
train['RUL'] = train['max_cycle'] - train['cycle']
train.drop(columns=['max_cycle'], inplace=True)
train['label'] = (train['RUL'] <= 30).astype(int)

print(f"Train shape: {train.shape}")

Train shape: (20631, 28)


In [2]:
#sensor yang std-nya nyaris 0 = todal omfprmatif = dibuang
flat_sensors = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']

train.drop(columns=flat_sensors, inplace = True)
test.drop(columns=flat_sensors, inplace= True)

print(f"Train shape sekarang: {train.shape}")
print(f"Test shape sekarang: {test.shape}")
print(f"\nKolom yang tersisa: {train.columns.tolist()}")

Train shape sekarang: (20631, 21)
Test shape sekarang: (13096, 19)

Kolom yang tersisa: ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3', 's2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21', 'RUL', 'label']


In [3]:
sensor_cols = [
    's2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21'
]

#inisialisasi scaler
scaler = MinMaxScaler()

#transform di train dan test, fit di train
train[sensor_cols] = scaler.fit_transform(train[sensor_cols])
test[sensor_cols] = scaler.transform(test[sensor_cols])

print("\nPrevier train setelah normalisasi")
train[sensor_cols].describe().round(3)


Previer train setelah normalisasi


,s2,s3,s4,s7,s8,s9,s11,s12,s13,s14,s15,s17,s20,s21
count,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000,20631.000
mean,0.443,0.425,0.450,0.566,0.298,0.195,0.411,0.581,0.318,0.226,0.451,0.434,0.524,0.546
std,0.151,0.134,0.152,0.143,0.108,0.099,0.159,0.157,0.106,0.098,0.144,0.129,0.140,0.149
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.336,0.332,0.339,0.477,0.227,0.141,0.298,0.484,0.235,0.172,0.346,0.333,0.434,0.452
50%,0.431,0.416,0.435,0.578,0.288,0.175,0.393,0.595,0.309,0.210,0.439,0.417,0.535,0.557
75%,0.539,0.509,0.545,0.670,0.364,0.214,0.506,0.695,0.382,0.250,0.541,0.500,0.628,0.653
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


In [4]:
#rolling window untuk tiap sensor
#tambah fitur: rata-rata 5 cycle terakhir dan std 5 cycle terakhir

window = 5

for sensor in sensor_cols:
    #rolling mean rata-rata 5 cycle
    train[f"{sensor}_rmean"] = (train.groupby('engine_id')[sensor].transform(lambda x: x.rolling(window, min_periods = 1).mean()))
    #rolling seberapa "goyang" sensor 5 cycle terakhir
    train[f"{sensor}_rstd"] = (train.groupby('engine_id')[sensor].transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0)))

print(f"Train shape sekarang: {train.shape}")
print(f"Jumlah fitur baru: {len(sensor_cols)*2}")

Train shape sekarang: (20631, 49)
Jumlah fitur baru: 28


In [5]:
#simpan data yang udah bersih
train.to_csv("../data/train_preprocessed.csv", index=False)
test.to_csv("../data/test_processed.csv", index = False)

print("Data tersimpan!")
print(f"Sensor tersisa : {len(sensor_cols)} sensor")
print(f"Fitur rolling dibuat {len(sensor_cols)*2} fitur")
print(f"Total sensor final : {train.shape[1]} kolom")
print(f"Total data train: {train.shape[0]} baris")

Data tersimpan!
Sensor tersisa : 14 sensor
Fitur rolling dibuat 28 fitur
Total sensor final : 49 kolom
Total data train: 20631 baris
